# 54 · Kiểm kê PadChest — bao nhiêu ảnh thật sự pretrain được

## Vì sao cần notebook này trước `41c`

`41a` đo được: ba CNN ImageNet **độc lập thật** với RAD-DINO (`corr 0.47–0.52` so với mốc
same-backbone `0.8236`) nhưng **quá yếu** (`0.149` vs `0.204`), nên ensemble ở mức probe **âm hết**
(`−0.0035` … `−0.0164`). Nút chặn là **độ mạnh**, không phải đa dạng.

PadChest sửa đúng chỗ đó, và nó **sạch với CXR-LT 2024**: PadChest là Hospital San Juan (Alicante),
MIMIC-CXR là Beth Israel (Boston) — không chung bệnh nhân, không chung ảnh, nên **không rò rỉ ảnh
test của ta**. Khác hẳn `torchxrayvision -mimic_*`.

## Và nó có thể mua nhiều hơn "một CNN mạnh hơn"

**17/30 nhãn của CXR-LT 2026 trùng tên với 40 nhãn của ta**, trong đó **4 là nhãn TAIL**:

```
                   n+ trong lat khop cua ta      co trong PadChest
Azygos Lobe                  176                       CO
Pneumoperitoneum             454                       CO
Hydropneumothorax            549                       CO
Kyphosis                     669                       CO
```

Toàn bộ khoảng cách `3.26×` tới vô địch nằm ở tail. Notebook này đo xem PadChest có **bao nhiêu ca
dương** cho đúng bốn nhãn đó.

## Cấu trúc thật của bản phân phối

Bản chia sẻ chính thức (`b2drop.bsc.es/nextcloud/s/BIMCV-PadChest-FULL`, **108 file, 964.1 GB**) có
**52 zip**, đánh số `0–50` và `54` — **`51`, `52`, `53` KHÔNG tồn tại**. README công khai của tác giả
ghi *"54 zip files"* là đếm không chính xác. Nên **không có gì bị thiếu**.

Mỗi zip kèm một `N.zip.unzip-l.txt` (đầu ra của `unzip -l`). Notebook này **đọc các file text đó**
thay vì mở mục lục zip: nhanh hơn hàng chục lần trên mount FUSE, và không cần chạm tới 964 GB.

## C0 · Config + liệt kê `PadChest_Full`

In [ ]:
import os, re, glob, json, zipfile, collections, time, io as _io
import numpy as np, pandas as pd

try:
    from google.colab import drive; drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/CXR-LT 2026'
except Exception:
    ROOT = './CXR-LT 2026'
PZ  = os.path.join(ROOT, 'PadChest_Full')
RES = os.path.join(ROOT, '_inventory')
def ensure_res():
    os.makedirs(RES, exist_ok=True); return RES
ensure_res()

P_TRAIN = os.path.join(ROOT, 'CXRLT_2026_training.csv')
P_REMOV = os.path.join(ROOT, 'Removal.csv')
P_CLASS = os.path.join(ROOT, 'Train_CLASS.txt')
for p in (PZ, P_TRAIN, P_REMOV, P_CLASS):
    assert os.path.exists(p), f'khong thay {p}'

# 40 nhan CXR-LT 2024 cua ta — de do phan trung
OUR40 = ['Adenopathy','Atelectasis','Azygos Lobe','Calcification of the Aorta','Cardiomegaly',
         'Clavicle Fracture','Consolidation','Edema','Emphysema','Enlarged Cardiomediastinum',
         'Fibrosis','Fissure','Fracture','Granuloma','Hernia','Hydropneumothorax','Infarction',
         'Infiltration','Kyphosis','Lobar Atelectasis','Lung Lesion','Lung Opacity','Mass','Nodule',
         'Normal','Pleural Effusion','Pleural Other','Pleural Thickening','Pneumomediastinum',
         'Pneumonia','Pneumoperitoneum','Pneumothorax','Pulmonary Embolism','Pulmonary Hypertension',
         'Rib Fracture','Round(ed) Atelectasis','Subcutaneous Emphysema','Support Devices',
         'Tortuous Aorta','Tuberculosis']
# 11 nhan TAIL cua ta (n+ < 1,000 tren lat khop 219,785) — do 52a
OUR_TAIL = {'Lobar Atelectasis':109, 'Clavicle Fracture':147, 'Round(ed) Atelectasis':147,
            'Azygos Lobe':176, 'Pneumoperitoneum':454, 'Pleural Other':530,
            'Hydropneumothorax':549, 'Infarction':592, 'Pneumomediastinum':624,
            'Kyphosis':669, 'Pulmonary Hypertension':749}

fs   = sorted(os.listdir(PZ))
zips = sorted([f for f in fs if f.lower().endswith('.zip')],
              key=lambda f: int(re.search(r'(\d+)', f).group(1)) if re.search(r'\d', f) else -1)
lsts = [f for f in fs if f.lower().endswith('.unzip-l.txt')]
oth  = [f for f in fs if f not in zips and f not in lsts]
_sz  = {f: os.path.getsize(os.path.join(PZ, f)) for f in zips}
print(f'{len(zips)} zip | {sum(_sz.values())/1e9:.1f} GB | {len(lsts)} file unzip-l.txt')
print(f'file khac: {oth}')
_id = sorted(int(re.search(r'(\d+)', f).group(1)) for f in zips)
print(f'\nchi so zip: {_id}')
_gap = sorted(set(range(min(_id), max(_id)+1)) - set(_id))
print(f'so KHUYET: {_gap}')
print('  -> ban chia se chinh thuc (108 file, 964.1 GB) nhay tu 50.zip sang 54.zip.')
print('     51/52/53 KHONG ton tai. Day la cau truc goc, KHONG phai tai thieu.')
print(f'\n{"zip":<10} {"GB":>7}   {"zip":<10} {"GB":>7}')
for i in range(0, len(zips), 2):
    a = f'{zips[i]:<10} {_sz[zips[i]]/1e9:>7.2f}'
    b = f'{zips[i+1]:<10} {_sz[zips[i+1]]/1e9:>7.2f}' if i+1 < len(zips) else ''
    print(f'  {a}   {b}')

## C1 · Kiểm kê ảnh — đọc `*.unzip-l.txt`, **không** mở zip

Mở mục lục zip đòi seek tới cuối file; với 52 file trên mount FUSE thì chậm và dễ treo (bài học
`33`: lặp trên mount treo cứng). Các file `unzip -l` đã có sẵn đúng thông tin đó ở dạng text.

Nếu một zip thiếu file `.txt` đi kèm thì mới mở mục lục zip cho riêng nó.

In [ ]:
_RX = re.compile(r'^\s*(\d+)\s+\d{2,4}[-/]\d{2}[-/]\d{2}\s+\d{2}:\d{2}\s+(\S.*?)\s*$')
IMG_EXT = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')

def from_txt(p):
    out = []
    with open(p, 'r', errors='ignore') as f:
        for ln in f:
            m = _RX.match(ln)
            if m and m.group(2).lower().endswith(IMG_EXT): out.append((int(m.group(1)), m.group(2)))
    return out

P_INV = os.path.join(RES, 'padchest_inventory.parquet')
if os.path.exists(P_INV):
    INV = pd.read_parquet(P_INV); print(f'nap lai {P_INV}: {len(INV):,} anh')
else:
    t0, rows, src = time.time(), [], {}
    for z in zips:
        t = os.path.join(PZ, z + '.unzip-l.txt')
        if os.path.exists(t):
            e = from_txt(t); src[z] = 'txt'
        else:                                   # du phong: mo muc luc zip
            with zipfile.ZipFile(os.path.join(PZ, z)) as zf:
                e = [(i.file_size, i.filename) for i in zf.infolist()
                     if not i.is_dir() and i.filename.lower().endswith(IMG_EXT)]
            src[z] = 'zip'
        rows += [(z, n, s) for s, n in e]
        print(f'  {z:<10} {len(e):>7,} anh  ({src[z]})', flush=True)
    INV = pd.DataFrame(rows, columns=['zip', 'path', 'bytes'])
    INV['name'] = INV.path.map(os.path.basename)
    ensure_res(); INV.to_parquet(P_INV, index=False)
    print(f'\n{len(INV):,} anh | {(time.time()-t0)/60:.1f} phut | ghi {P_INV}')
print(f'\ntong {len(INV):,} anh | {INV.bytes.sum()/1e9:.0f} GB giai nen | '
      f'{len(INV.name.unique()):,} ten DUY NHAT')
assert len(INV) == len(INV.name.unique()), 'co ten anh TRUNG giua cac zip -> kiem lai'
print('duoi:', collections.Counter(os.path.splitext(n)[1].lower()
                                   for n in INV.name.head(50000)).most_common(4))
print('vi du:', INV.path.head(3).tolist())

# doi chieu voi Verify_Zips_ImageCounts neu co
_v = [f for f in oth if 'verify' in f.lower()]
if _v:
    try:
        V = (pd.read_excel(os.path.join(PZ, _v[0])) if _v[0].lower().endswith(('.xlsx', '.xls'))
             else pd.read_csv(os.path.join(PZ, _v[0])))
        print(f'\n--- {_v[0]} ---'); print(V.head(10).to_string(index=False))
    except Exception as e:
        print(f'\nkhong doc duoc {_v[0]}: {type(e).__name__}: {e}')

## C2 · Độ phủ — bao nhiêu ảnh training của CXR-LT 2026 thật sự có trên đĩa

In [ ]:
T = pd.read_csv(P_TRAIN, low_memory=False)
R = pd.read_csv(P_REMOV, low_memory=False)
CL = [l.strip() for l in open(P_CLASS, encoding='utf-8') if l.strip()]
print(f'training {len(T):,} hang x {T.shape[1]} cot | removal {len(R):,} | {len(CL)} lop')
print('cot training:', list(T.columns)[:20])

k = next((c for c in T.columns if re.search(r'imageid|image_id|filename|^path$|^image$',
                                            str(c), re.I)), T.columns[0])
rk = next((c for c in R.columns if str(c).lower() == str(k).lower()), R.columns[0])
print(f'cot khoa: training[{k}]  vs  removal[{rk}]')
base = lambda s: os.path.basename(str(s)).strip()
T['_b'] = T[k].map(base); R['_b'] = R[rk].map(base)
HAVE = set(INV.name)
T['_keep'] = ~T._b.isin(set(R._b))
T['_have'] = T._b.isin(HAVE)
T['_ok']   = T._keep & T._have
n_ok = int(T._ok.sum())
print(f'\n{"":<28}{"so anh":>10}  {"ty le":>7}')
print(f'{"training goc":<28}{len(T):>10,}')
print(f'{"sau khi tru Removal":<28}{int(T._keep.sum()):>10,}  {T._keep.mean():>7.1%}')
print(f'{"CO anh tren dia":<28}{int(T._have.sum()):>10,}  {T._have.mean():>7.1%}')
print(f'{"DUNG DUOC (ca hai)":<28}{n_ok:>10,}  {T._ok.mean():>7.1%}   <- N pretrain')
print(f'{"thieu anh":<28}{int((T._keep & ~T._have).sum()):>10,}')

# --- cong: neu thieu qua nhieu thi phai xem lai truoc khi dung ---
_miss = 1 - T._have.mean()
print(f'\ncong: thieu {_miss:.1%}')
if _miss > 0.20:
    print('  !! THIEU > 20% -> kiem lai cot khoa hoac tai bu. DUNG chay 41c voi tap nay.')
elif _miss > 0.10:
    print('  vung xam 10-20% -> chay duoc, nhung C3 phai kiem 4 nhan TAIL co bi mat can xung khong')
else:
    print('  < 10% -> chay duoc, ghi ro "n/142,928" trong REPORT')
ensure_res()
T.loc[T._ok, [k, '_b'] + [c for c in T.columns if c in CL]].to_parquet(
    os.path.join(RES, 'padchest_usable.parquet'), index=False)
print(f'ghi {os.path.join(RES, "padchest_usable.parquet")}')

## C3 · Nhãn — trùng bao nhiêu với 40 nhãn của ta, và **tail** được bao nhiêu ca dương

Đây là cột quyết định giá trị của cả hướng. Nếu 4 nhãn tail trùng chỉ có vài chục ca dương thì
PadChest chỉ đóng khoảng cách miền; nếu có hàng nghìn thì nó là **nguồn giám sát ngoài cho đúng chỗ
ta thiếu nhất**.

In [ ]:
U = T[T._ok]
def npos(c):
    col = next((x for x in U.columns if str(x).strip().lower() == c.strip().lower()), None)
    if col is None: return -1
    return int(pd.to_numeric(U[col], errors='coerce').fillna(0).gt(0.5).sum())
OUR = set(OUR40)
rows = [dict(lop=c, n_duong=npos(c), trung_40=c in OUR, tail_cua_ta=OUR_TAIL.get(c, 0))
        for c in CL]
D = pd.DataFrame(rows).sort_values('n_duong', ascending=False)
print(f'{"lop":<32} {"n duong":>9} {"ty le":>7}  trung 40   n+ cua ta (neu tail)')
for r in D.itertuples():
    _t = f'{r.tail_cua_ta:,}' if r.tail_cua_ta else ''
    print(f'{r.lop:<32} {r.n_duong:>9,} {r.n_duong/max(len(U),1):>7.2%}  '
          f'{"<<<" if r.trung_40 else "   ":<8} {_t}')
_ov = D[D.trung_40]
print(f'\n{len(_ov)}/{len(CL)} lop trung ten voi 40 nhan CXR-LT 2024')

print('\n=== 4 NHAN TAIL CUA TA CO TRONG PADCHEST ===')
print(f'{"nhan":<24} {"n+ cua ta":>10} {"n+ PadChest":>12} {"boi so":>8}')
for c, n_ours in sorted(OUR_TAIL.items(), key=lambda x: x[1]):
    n_pc = npos(c)
    if n_pc < 0: continue
    print(f'{c:<24} {n_ours:>10,} {n_pc:>12,} {n_pc/max(n_ours,1):>7.1f}x')
print('  boi so > 1 => PadChest co NHIEU ca duong hon lat khop cua ta cho nhan do.')
print('  Do la nguon giam sat NGOAI cho dung cho khoang cach 3.26x toi vo dich.')
ensure_res(); D.to_csv(os.path.join(RES, 'padchest_labels.csv'), index=False)

## C4 · Một ảnh mẫu — độ phân giải, dtype, và ước lượng chi phí stage

In [ ]:
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
z0 = zips[0]
with zipfile.ZipFile(os.path.join(PZ, z0)) as zf:
    _cand = [n for n in zf.namelist() if n.lower().endswith(IMG_EXT)][:5]
    print(f'doc {len(_cand)} anh mau tu {z0}')
    for n in _cand:
        b = zf.read(n); im = Image.open(_io.BytesIO(b)); a = np.asarray(im)
        print(f'  {os.path.basename(n)[:34]:<36} {im.size} {im.mode} {a.dtype} '
              f'min={a.min()} max={a.max()} | {len(b)/1e6:.1f} MB')
_bit = '16-bit' if a.dtype != np.uint8 else '8-bit'
print(f'\n=> {_bit}. Chuyen doi sang JPEG 518px grayscale:')
print(f'   dia sau khi resize : ~{n_ok * 55e3/1e9:.0f} GB   ({n_ok:,} anh x ~55 KB)')
print(f'   thoi gian stage    : ~{n_ok/200/60:.0f}-{n_ok/80/60:.0f} phut '
      '(giai nen + resize, 8 luong)')
print(f'   doc tu Drive       : {INV.bytes.sum()/1e9:.0f} GB nen / 83 MB/s '
      f'~= {INV.bytes.sum()/83e6/3600:.1f} gio NEU doc het')
print('\n   -> `41c` chi can doc cac zip CHUA anh dung duoc. Neu anh nam rai deu 52 zip thi')
print('      van phai doc gan het => day la RANG BUOC THAT SU cua ke hoach, khong phai GPU.')
_zc = INV[INV.name.isin(set(T.loc[T._ok, '_b']))].zip.value_counts()
print(f'\n   anh dung duoc nam o {len(_zc)}/{len(zips)} zip | '
      f'{_zc.head(5).to_dict()}')

## C5 · Ghi `REVIEW_PADCHEST.md`

In [ ]:
L = []; w = L.append
w('# REVIEW · kiểm kê PadChest cho pretrain nhánh CNN'); w('')
w(f'`{PZ}` · **{len(zips)} zip**, `{sum(_sz.values())/1e9:.0f}` GB nén, `{len(INV):,}` ảnh.')
w(f'Chỉ số zip `{min(_id)}–{max(_id)}`, khuyết `{_gap}` — **cấu trúc gốc của bản phân phối chính '
  'thức** (`b2drop.bsc.es`, 108 file, 964.1 GB, nhảy từ `50.zip` sang `54.zip`), không phải tải '
  'thiếu.'); w('')
w('| | số ảnh | tỉ lệ |'); w('|---|---|---|')
w(f'| training gốc (CXR-LT 2026) | {len(T):,} | — |')
w(f'| sau khi trừ `Removal.csv` | {int(T._keep.sum()):,} | {T._keep.mean():.1%} |')
w(f'| có ảnh trên đĩa | {int(T._have.sum()):,} | {T._have.mean():.1%} |')
w(f'| **dùng được** | **{n_ok:,}** | **{T._ok.mean():.1%}** |'); w('')
w(f'`{len(_ov)}/{len(CL)}` lớp trùng tên với 40 nhãn CXR-LT 2024, trong đó **4 là nhãn tail** của ta:')
w(''); w('```')
w(f'{"nhan":<24} {"n+ cua ta":>10} {"n+ PadChest":>12}')
for c, n_ours in sorted(OUR_TAIL.items(), key=lambda x: x[1]):
    n_pc = npos(c)
    if n_pc >= 0: w(f'{c:<24} {n_ours:>10,} {n_pc:>12,}')
w('```'); w('')
w('## Vì sao PadChest sạch với CXR-LT 2024'); w('')
w('PadChest là Hospital San Juan (Alicante), MIMIC-CXR là Beth Israel (Boston) — **không chung bệnh '
  'nhân, không chung ảnh**. Split test của CXR-LT 2024 là tập con của MIMIC-CXR, nên pretrain trên '
  'PadChest **không** làm lộ ảnh test. Khác hẳn `torchxrayvision -mimic_*`, thứ đã thấy ảnh test '
  'của ta **kèm nhãn**.'); w('')
w('`Removal.csv` do ban tổ chức cung cấp đã loại sẵn ảnh dành cho dev/test của **2026**, nên tập '
  'này cũng sạch với 2026.'); w('')
w('## Những gì kết quả này KHÔNG trả lời'); w('')
w('- **Không đo được `corr` sau pretrain.** Kéo CNN về miền CXR là kéo nó **về gần** RAD-DINO. '
  f'`41a` đo `corr 0.47–0.52` trên đặc trưng ImageNet, so với mốc same-backbone `0.8236`. Sau '
  'pretrain con số đó **sẽ tăng** — phải đo lại, không được mang sang dùng.')
w('- **Ràng buộc giấy phép.** Nhãn CXR-LT 2026 *"may not be shared under any circumstances"* và '
  'phải xoá sau `04/11/2026`; luật thi cũng ghi *"may only train your models on the 30 labels"*. '
  'Việc dùng chúng để pretrain cho một **giải khác** là câu hỏi cho ban tổ chức, không phải cho '
  'notebook này.')
w('- **Ràng buộc thật là I/O, không phải GPU.** Đọc hết zip từ Drive ở `83 MB/s` mất '
  f'`~{INV.bytes.sum()/83e6/3600:.0f}` giờ. Đó mới là con số quyết định `41c` có kịp hay không.')
TXT = '\n'.join(L); ensure_res()
open(os.path.join(RES, 'REVIEW_PADCHEST.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)